In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib_venn import venn2
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
def read_te_annotation(file):
    column_names = [
        "Repeat_ID",
        "Divergence",
        "Deletion",
        "Insertion",
        "Sequence_Name",
        "Alignment_Start",
        "Alignment_End",
        "Element_Length",
        "Orientation",
        "Family_Name",
        "Class",
        "Base_Count",
        "5'_Fragment_Length",
        "3'_Fragment_Length",
        "Additional_Info",
        "Repeat_Count",
        "Score"
    ]
    df = pd.read_csv(file, delim_whitespace=True, comment='#', header=None, names=column_names)
    return df

In [4]:
def extract_gene_name(attributes):
    desc = attributes.split(';')
    gene_name = desc[2].split('=')[1].lower()
    return gene_name

In [5]:
def read_gff(path, skiprows):
    gff_columns = [
    "seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"
    ]
    annotation = pd.read_csv(path, sep='\t', skiprows=9, names=gff_columns)
    annotation_filtered = annotation[annotation['type'] == 'gene']
    #for index, row in annotation_filtered.iterrows():
    #    desc = row['attributes'].split(';')
    #    gene_name = desc[2].split('=')[1].lower()
    #      annotation[index]['gene_name'] = gene_name
    annotation_filtered['gene_name'] = annotation_filtered['attributes'].apply(extract_gene_name)
    return annotation_filtered

In [6]:
def find_te_overlaps(te_df, gene_df, upstream=10000, downstream=10000):
    # Crear una lista para almacenar los resultados
    overlap_results = []

    # Iterar sobre cada gen en el DataFrame de genes
    for index, gene_row in gene_df.iterrows():
        gene_name = gene_row['gene_name']
        gene_seqid = gene_row['seqid']
        gene_start = gene_row['start']
        gene_end = gene_row['end']
        gene_strand = gene_row['strand']
        attributes = gene_row['attributes']
        # Filtrar los TEs que están en el mismo cromosoma o secuencia que el gen
        relevant_tes = te_df[te_df['Sequence_Name'] == gene_seqid]

        # Inicializar los valores para family_name, class y tipo de overlap
        te_family_name = np.nan
        te_class = np.nan
        overlap_type = np.nan

        # Buscar elementos transponibles que hagan solapamiento
        for _, te_row in relevant_tes.iterrows():
            te_start = te_row['Alignment_Start']
            te_end = te_row['Alignment_End']
            te_family = te_row['Family_Name']
            te_class_type = te_row['Class']

            # Verificar si hay solapamiento instream
            if (te_start >= gene_start and te_start <= gene_end) or (te_end >= gene_start and te_end <= gene_end):
                te_family_name = te_family
                te_class = te_class_type
                overlap_type = "instream"
                break

            # Verificar si hay solapamiento upstream
            elif (gene_start - upstream <= te_end <= gene_start):
                te_family_name = te_family
                te_class = te_class_type
                overlap_type = "upstream"
                break

            # Verificar si hay solapamiento downstream
            elif (gene_end <= te_start <= gene_end + downstream):
                te_family_name = te_family
                te_class = te_class_type
                overlap_type = "downstream"
                break

        # Agregar el resultado a la lista
        overlap_results.append({
            'seqid': gene_seqid,
            'gene_name': gene_name,
            'gene_start': gene_start,
            'gene_end': gene_end,
            'strand': gene_strand,
            'te_family_name': te_family_name,
            'te_class': te_class,
            'overlap_type': overlap_type,
            'attributes': attributes
        })

    # Convertir la lista de resultados en un DataFrame
    overlap_df = pd.DataFrame(overlap_results)
    return overlap_df

# Suzukii

In [10]:
suzukii_tes = read_te_annotation('suzukii_GCF_043229965.1_octfa.out')
suzukii_tes.to_csv("suzukii_te_annotation.tsv", sep='\t',index=False)
suzukii_tes.head()

C:\Users\JuanG\AppData\Local\Temp\ipykernel_16700\3938828656.py:21: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file, delim_whitespace=True, comment='#', header=None, names=column_names)


,Repeat_ID,Divergence,Deletion,Insertion,Sequence_Name,Alignment_Start,Alignment_End,Element_Length,Orientation,Family_Name,Class,Base_Count,5'_Fragment_Length,3'_Fragment_Length,Additional_Info,Repeat_Count,Score
0,10299,1.300,3.000,0.600,NC_092080.1,21824,23072,1269,C,nf_292_LTR_1,LTR/LTR,(0),1269,1,28,1,No_ref_available
1,2536/284,7.711,2.253,1.259,NC_092080.1,35610,44782,402,C,nf_26_COPIA_1,LTR/COPIA,(352),8153,4236,43/58,2,No_ref_available
2,886/366,16.849,0.000,0.000,NC_092080.1,35924,36254,214,+,nf_25_CR1_1,LINE/CR1,1565,1779,(6831),44/44,2,0.025
3,523,19.000,18.100,1.200,NC_092080.1,36339,36482,168,+,nf_205_CR1_2,LINE/CR1,1763,1930,(37),46,1,0.085
4,578,7.200,12.100,1.800,NC_092080.1,36975,37073,109,C,Gypsy-30_DWil-I_1,LTR/GYPSY,(261),5618,5510,48,1,No_ref_available


In [21]:
suzukii_te_annotation = pd.read_csv("suzukii_te_annotation.tsv", sep="\t")

suzukii_gff_df = pd.DataFrame({
    "seqid": suzukii_te_annotation["Sequence_Name"],
    "source": "RepeatMasker",
    "type": "transposable_element",
    "start": suzukii_te_annotation["Alignment_Start"],
    "end": suzukii_te_annotation["Alignment_End"],
    "score": ".",
    "strand": suzukii_te_annotation["Orientation"].replace({"C": "-"}),
    "phase": ".",
    "attributes": "ID=" + suzukii_te_annotation["Family_Name"] + "_" + suzukii_te_annotation.index.astype(str) +
                  ";Family=" + suzukii_te_annotation["Family_Name"] +
                  ";Class=" + suzukii_te_annotation["Class"]
})

suzukii_gff_df.to_csv("suzukii_te_annotation.gff", sep="\t", header=False, index=False)

In [7]:
suzukii_gff = read_gff('suzukii_transfer_annotation_v2.gff', 3)
suzukii_gff.head()

C:\Users\JuanG\AppData\Local\Temp\ipykernel_16600\2124761365.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotation_filtered['gene_name'] = annotation_filtered['attributes'].apply(extract_gene_name)


,seqid,source,type,start,end,score,strand,phase,attributes,gene_name
1,NC_060762.1,Liftoff,gene,1480,3015,.,+,.,ID=gene-Dmel_CG34067;Dbxref=FLYBASE:FBgn001367...,cox1
5,NC_060762.1,Liftoff,gene,3018,3083,.,+,.,ID=gene-Dmel_CR34068;Dbxref=FLYBASE:FBgn001369...,trnl1
8,NC_060762.1,Liftoff,gene,3089,3773,.,+,.,ID=gene-Dmel_CG34069;Dbxref=FLYBASE:FBgn001367...,cox2
12,NC_060762.1,Liftoff,gene,3774,3844,.,+,.,ID=gene-Dmel_CR34070;Dbxref=FLYBASE:FBgn001369...,trnk
15,NC_060762.1,Liftoff,gene,4069,4743,.,+,.,ID=gene-Dmel_CG34073;Dbxref=FLYBASE:FBgn001367...,atp6


In [ ]:
overlap_df_suzukii = find_te_overlaps(suzukii_tes, suzukii_gff)
overlap_df_suzukii.head(1000)

# Melanogaster

In [22]:
melanogaster_tes = read_te_annotation('melano_octfa.out')
#melanogaster_tes['Class'] = melanogaster_tes['Class'].replace('LTR/GYPSY', 'LTR/Gypsy')
#melanogaster_tes['Class'] = melanogaster_tes['Class'].replace('LTR/BELPAO', 'LTR/BEL-PAO')
#melanogaster_tes['Class'] = melanogaster_tes['Class'].replace('DNA/TC1MARINER', 'DNA/TC1-MARINER')
#melanogaster_tes['Class'] = melanogaster_tes['Class'].str.upper()
melanogaster_tes.to_csv("melanogaster_te_annotation.tsv", sep='\t',index=False)
melanogaster_tes.head()

C:\Users\JuanG\AppData\Local\Temp\ipykernel_16700\3938828656.py:21: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file, delim_whitespace=True, comment='#', header=None, names=column_names)


,Repeat_ID,Divergence,Deletion,Insertion,Sequence_Name,Alignment_Start,Alignment_End,Element_Length,Orientation,Family_Name,Class,Base_Count,5'_Fragment_Length,3'_Fragment_Length,Additional_Info,Repeat_Count,Score
0,872,7.000,4.700,0.800,NC_004353.4,1563,1691,134,+,DNAREP1_DM,DNA/RC,572,705.0,(0),5,1,0.190
1,920/355,16.639,25.066,1.431,NC_004353.4,5057,5492,494,+,DNAREP1_DM,DNA/RC,3,705.0,(0),6/6,2,0.701
2,590,13.000,0.000,5.000,NC_004353.4,7131,7235,100,C,DNAREP1_DM,DNA/RC,(541),164.0,65,8,1,0.142
3,937/355,16.413,25.066,1.431,NC_004353.4,11838,12273,494,+,DNAREP1_DM,DNA/RC,3,705.0,(0),9/9,2,0.701
4,55699,0.300,1.400,0.200,NC_004353.4,13649,20087,6519,+,Quasimodo2_DM,LTR/GYPSY,1,6519.0,(0),11,1,No_ref_available


In [ ]:
melanogaster_gff = read_gff('melano.gff', 8)
melanogaster_gff.head()

In [24]:
melano_te_annotation = pd.read_csv("melanogaster_te_annotation.tsv", sep="\t")

melano_gff_df = pd.DataFrame({
    "seqid": melano_te_annotation["Sequence_Name"],
    "source": "RepeatMasker",
    "type": "transposable_element",
    "start": melano_te_annotation["Alignment_Start"],
    "end": melano_te_annotation["Alignment_End"],
    "score": ".",
    "strand": melano_te_annotation["Orientation"].replace({"C": "-"}),
    "phase": ".",
    "attributes": "ID=" + melano_te_annotation["Family_Name"] + "_" + melano_te_annotation.index.astype(str) +
                  ";Family=" + melano_te_annotation["Family_Name"] +
                  ";Class=" + melano_te_annotation["Class"]
})

melano_gff_df.to_csv("melanogaster_te_annotation.gff", sep="\t", header=False, index=False)

In [ ]:
overlap_df_melano = find_te_overlaps(melanogaster_tes, melanogaster_gff)
overlap_df_melano.head(50)

In [ ]:
comparison_df=pd.read_csv("comparison_df.csv")

In [ ]:
def compare_species_overlaps(suzukii_df, melano_df):
    # Identificar genes comunes
    common_genes = pd.merge(suzukii_df[['gene_name']], melano_df[['gene_name']], on='gene_name', how='inner')

    # Unir los DataFrames basados en el nombre del gen
    merged_df = pd.merge(suzukii_df, melano_df, on='gene_name', suffixes=('_suzukii', '_melano'))

    # Crear columnas para comparar TEs
    merged_df['te_family_diff'] = merged_df['te_family_name_suzukii'] != merged_df['te_family_name_melano']
    merged_df['te_class_diff'] = merged_df['te_class_suzukii'] != merged_df['te_class_melano']
    merged_df['overlap_type_diff'] = merged_df['overlap_type_suzukii'] != merged_df['overlap_type_melano']

    # Crear una columna indicando si hay alguna diferencia
    merged_df['has_difference'] = merged_df[['te_family_diff', 'te_class_diff', 'overlap_type_diff']].any(axis=1)

    # Eliminar filas donde no hay ningún overlap
    merged_df = merged_df.dropna(subset=['overlap_type_suzukii', 'overlap_type_melano'], how='all')

    return merged_df

In [ ]:
# Usar la función
comparison_df = compare_species_overlaps(suzukii_df=overlap_df_suzukii, melano_df=overlap_df_melano)
# Mostrar las diferencias
#comparison_df[comparison_df['has_difference']].head(50)

In [ ]:
# Supongamos que tu DataFrame se llama 'df'
comparison_df.to_csv('comparison_df.csv', index=False)

In [ ]:
# Contar las diferencias
diff_counts = comparison_df['has_difference'].value_counts()
print(diff_counts)

In [ ]:
"""
def create_heatmap(comparison_df):
    #comparison_df['suzukii_te_present'] = comparison_df['te_family_name_suzukii'].apply(lambda x: 1 if x else 0)
    comparison_df['melano_te_present'] = comparison_df['te_family_name_melano'].apply(lambda x: 1 if x else 0)

    # Crear el heatmap
    heatmap_data = comparison_df[['gene_name', 'suzukii_te_present', 'melano_te_present']]
    heatmap_data.set_index('gene_name', inplace=True)

    plt.figure(figsize=(10, 14))
    sns.heatmap(heatmap_data, cmap='coolwarm', cbar_kws={'label': 'TE Presence'})
    plt.title('TE Presence/Absence in D. suzukii and D. melanogaster Genes')
    plt.xlabel('Species')
    plt.ylabel('Genes')
    plt.show()
"""

In [ ]:
#create_heatmap(comparison_df)

In [ ]:
def create_venn_diagram(comparison_df):

    # Conjuntos de genes con TEs en cada especie
    suzukii_genes_with_tes = set(comparison_df[comparison_df['te_family_name_suzukii'] != '']['gene_name'])
    melano_genes_with_tes = set(comparison_df[comparison_df['te_family_name_melano'] != '']['gene_name'])

    # Crear el gráfico de Venn
    plt.figure(figsize=(8, 8))
    venn2([suzukii_genes_with_tes, melano_genes_with_tes], set_labels=('D. suzukii', 'D. melanogaster'))
    plt.title('Gráfico de Venn de Genes con TEs en D. suzukii y D. melanogaster')
    plt.show()

In [ ]:
#create_venn_diagram(comparison_df)

In [ ]:
def create_stacked_bar_chart(comparison_df):
    # Count occurrences by overlap type in each species
    overlap_counts_suzukii = comparison_df['overlap_type_suzukii'].value_counts()
    overlap_counts_melano = comparison_df['overlap_type_melano'].value_counts()

    # Create DataFrame for stacked bar chart
    overlap_df = pd.DataFrame({
        'D. suzukii': overlap_counts_suzukii,
        'D. melanogaster': overlap_counts_melano
    }).fillna(0)

    # Create the stacked bar chart
    ax = overlap_df.plot(kind='bar', stacked=True, figsize=(12, 8), color=['#1f77b4', '#ff7f0e'])

    # Set the title and labels
    plt.title('Overlap Type Distribution of TEs in Genes of D. suzukii and D. melanogaster')
    plt.xlabel('Overlap Type')
    plt.ylabel('Number of Genes')
    plt.legend(title='Species')

    # Adjust layout to prevent cutting off x-axis labels
    plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better visibility
    plt.tight_layout()  # Adjust layout to fit everything properly

    # Save the plot to the specified output path
    plt.savefig('overlap_type_distribution.png', format='png', dpi=300)
    plt.show()

In [ ]:
def create_heatmap(comparison_df):
    # Count occurrences by overlap type in each species
    overlap_counts_suzukii = comparison_df['overlap_type_suzukii'].value_counts()
    overlap_counts_melano = comparison_df['overlap_type_melano'].value_counts()

    # Crear un DataFrame para el mapa de calor
    overlap_df = pd.DataFrame({
        r'$\mathit{D. \ suzukii}$': overlap_counts_suzukii,
        r'$\mathit{D. \ melanogaster}$': overlap_counts_melano
    }).fillna(0)

    # Crear el mapa de calor con formato sin notación científica
    plt.figure(figsize=(8, 6))
    sns.heatmap(overlap_df, annot=True, fmt='g', cmap='YlGnBu', cbar_kws={'label': 'Number of Genes'})

    # Configurar título y etiquetas
    #plt.title('Heatmap of Overlap Type Distribution of TEs in Genes of D. suzukii and D. melanogaster', pad=20)
    plt.xlabel('Species')
    plt.ylabel('Overlap Type')

    # Ajustar el diseño para que todo encaje correctamente
    plt.tight_layout()

    # Reservar espacio adicional para el título
    plt.subplots_adjust(top=0.85)

    # Guardar la gráfica en el camino de salida especificado
    plt.savefig('overlap_type_heatmap.png', format='png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()  # Close the plot to free up memory

In [ ]:
create_stacked_bar_chart(comparison_df)

In [ ]:
create_heatmap(comparison_df)

In [ ]:
temp = comparison_df[
        (comparison_df['overlap_type_suzukii'] != '') &
        (comparison_df['overlap_type_melano'] != '')
    ]
temp = temp.dropna(subset=['overlap_type_suzukii', 'overlap_type_melano'])
print(len(list(set(temp['gene_name'].tolist()))))

In [ ]:
def perform_clustering(comparison_df, n_clusters=3):
    # Filtrar el DataFrame para excluir filas vacías o NaN
    filtered_comparison_df = comparison_df.dropna(subset=['overlap_type_suzukii', 'overlap_type_melano'])
    filtered_comparison_df = filtered_comparison_df[
        (filtered_comparison_df['overlap_type_suzukii'] != '') &
        (filtered_comparison_df['overlap_type_melano'] != '')
    ]

    # Crear columnas binarias para presencia (1) o ausencia (0) de TEs
    filtered_comparison_df['suzukii_te_present'] = filtered_comparison_df['te_family_name_suzukii'].apply(lambda x: 1 if x else 0)
    filtered_comparison_df['melano_te_present'] = filtered_comparison_df['te_family_name_melano'].apply(lambda x: 1 if x else 0)

    # Seleccionar las columnas relevantes para el clustering
    features = filtered_comparison_df[['suzukii_te_present', 'melano_te_present']]

    # Aplicar K-means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    filtered_comparison_df['cluster'] = kmeans.fit_predict(features)

    # Visualizar con un gráfico de dispersión de PCA (análisis de componentes principales)
    pca = PCA(n_components=2)
    components = pca.fit_transform(features)
    filtered_comparison_df['PCA1'] = components[:, 0]
    filtered_comparison_df['PCA2'] = components[:, 1]

    plt.figure(figsize=(10, 8))
    sns.scatterplot(x='PCA1', y='PCA2', hue='cluster', data=filtered_comparison_df, palette='viridis')
    plt.title('Clustering de Genes Basado en la Presencia de TEs')
    plt.xlabel('Componente Principal 1')
    plt.ylabel('Componente Principal 2')
    plt.show()

    return filtered_comparison_df

In [ ]:
#clustered_df = perform_clustering(comparison_df)

In [ ]:
def plot_genes_by_te_class(merged_df):
    # Count the number of genes by TE class for each genome
    suzukii_class_counts = merged_df.groupby('te_class_suzukii')['gene_name'].nunique()
    melano_class_counts = merged_df.groupby('te_class_melano')['gene_name'].nunique()

    # Create a DataFrame with the counts
    class_counts_df = pd.DataFrame({
        'D. suzukii': suzukii_class_counts,
        'D. melanogaster': melano_class_counts
    }).fillna(0)  # Fill with 0 for classes not present in a genome

    # Create a stacked bar chart
    class_counts_df.plot(kind='bar', stacked=True, figsize=(12, 8), color=['#1f77b4', '#ff7f0e'])

    # Set titles and labels
    plt.title('Number of Genes with TE Presence by Superfamily in D. suzukii and D. melanogaster')
    plt.xlabel('Superfamily')
    plt.ylabel('Number of Genes')
    plt.legend(title='Species')
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    # Save the plot to the specified output path
    plt.savefig('genes_with_te_by_superfamily.png', format='png', dpi=300)
    plt.show()

In [ ]:
def plot_te_class_heatmap(merged_df):
    """
    Plots a heatmap to visualize the number of genes with TE presence by superfamily
    in D. suzukii and D. melanogaster.

    Parameters:
    merged_df (DataFrame): A DataFrame containing the data with columns for TE classes and gene names.
    """
    # Count the number of genes by TE class for each genome
    suzukii_class_counts = merged_df.groupby('te_class_suzukii')['gene_name'].nunique()
    melano_class_counts = merged_df.groupby('te_class_melano')['gene_name'].nunique()

    # Create a DataFrame with the counts
    class_counts_df = pd.DataFrame({
        r'$\mathit{D. \ suzukii}$': suzukii_class_counts,
        r'$\mathit{D. \ melanogaster}$': melano_class_counts
    }).fillna(0)  # Fill with 0 for classes not present in a genome

    # Create the heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(class_counts_df, annot=True, fmt='g', cmap='YlGnBu', cbar_kws={'label': 'Number of Genes'})

    # Set titles and labels
    #plt.title('Heatmap of Number of Genes with TE Presence by Superfamily in D. suzukii and D. melanogaster')
    plt.xlabel('Species')
    plt.ylabel('Superfamily')

    # Adjust layout to fit everything properly
    plt.tight_layout()

    # Save the plot to the specified output path
    plt.savefig('genes_with_te_heatmap.png', format='png', dpi=300)
    plt.show()

In [ ]:
# Usar la función
plot_genes_by_te_class(comparison_df)

In [ ]:
plot_te_class_heatmap(comparison_df)

In [ ]:
def plot_overlaps_by_gene(merged_df):
    # Filtrar para eliminar filas sin solapamientos
    overlap_df = merged_df.dropna(subset=['overlap_type_suzukii', 'overlap_type_melano'], how='all')

    # Crear un DataFrame con las columnas relevantes
    overlaps_data = overlap_df[['gene_name', 'overlap_type_suzukii', 'overlap_type_melano', 'te_class_suzukii', 'te_class_melano']]

    # Preparar los datos para el gráfico
    overlaps_data = overlaps_data.melt(id_vars='gene_name', value_vars=['overlap_type_suzukii', 'overlap_type_melano'],
                                       var_name='species', value_name='overlap_type')

    # Eliminar filas donde no hay tipo de solapamiento (overlap)
    overlaps_data = overlaps_data.dropna(subset=['overlap_type'])

    # Crear el gráfico de barras múltiples
    plt.figure(figsize=(14, 8))
    sns.countplot(data=overlaps_data, x='gene_name', hue='overlap_type', dodge=True)

    # Configurar los títulos y etiquetas
    plt.title('Tipo de Solapamiento para Cada Gen en D. suzukii y D. melanogaster')
    plt.xlabel('Nombre del Gen')
    plt.ylabel('Cantidad de Solapamientos')
    plt.xticks(rotation=90)
    plt.legend(title='Tipo de Solapamiento')
    plt.tight_layout()
    plt.show()

In [ ]:
#plot_overlaps_by_gene(comparison_df)

In [ ]:
comparison_final = pd.read_csv('comparison_df.csv')
comparison_final.head()

In [ ]:
# para el panel A de la figura 5
from scipy.stats import chi2_contingency

# Construye la tabla de contingencia
#               Upstream  Downstream  Instream
table = [[3393,     1486,       396],      # D. suzukii
         [1723,     1177,       488]]      # D. melanogaster

chi2, p, dof, expected = chi2_contingency(table)
print(f"Chi² = {chi2:.2f}, p-value = {p:.4f}")

from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

# Datos de la figura
counts_suz = [3393, 1486, 396]
counts_mel = [1723, 1177, 488]
totals_suz = sum(counts_suz)
totals_mel = sum(counts_mel)

# Totales por categoría
total_per_type = [s + m for s, m in zip(counts_suz, counts_mel)]

# Comparaciones por tipo de solapamiento
pvals = []
for suz, mel, total in zip(counts_suz, counts_mel, total_per_type):
    count = [suz, mel]
    nobs = [total, total]  # misma base para comparar proporciones entre especies
    stat, pval = proportions_ztest(count, nobs)
    pvals.append(pval)

# Corrección por comparaciones múltiples
reject, pvals_corrected, _, _ = multipletests(pvals, method='bonferroni')

# Mostrar resultados
labels = ['Upstream', 'Downstream', 'Instream']
for i in range(3):
    print(f"{labels[i]}: p = {pvals[i]:.4f}, p_corr = {pvals_corrected[i]:.4f}, significant: {reject[i]}")


In [ ]:
# para el panel B de la figura 5
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

# Crear el DataFrame con tus datos
data = {
    "Superfamily": [
        "DNA/CMC", "DNA/DNA", "DNA/MITE", "DNA/Maverick",
        "DNA/P", "DNA/PIF-Harbinger", "DNA/PiggyBac", "DNA/RC", "DNA/Tc1-Mariner",
        "DNA/Transib", "DNA/hAT", "LINE/CR1", "LINE/I", "LINE/Jockey", "LINE/L2",
        "LINE/LINE", "LINE/R1", "LTR/Bel-Pao", "LTR/Copia","LTR/Gypsy"],
    "D. suzukii": [
        29, 1059, 29, 39, 96,
        67, 1, 841, 104, 2,
        71, 92, 167, 49, 6, 19,
        24, 49, 94, 399
    ],
    "D. melanogaster": [
        0, 63, 8, 0, 34,
        0, 0, 431, 92, 2,
        17, 5, 9, 155, 0, 45,
        15, 242, 27, 337,
    ]
}
df = pd.DataFrame(data)

# ❗ Eliminar filas donde ambas especies tienen cero
df_filtered = df[(df["D. suzukii"] > 0) | (df["D. melanogaster"] > 0)]

# 🔹 Prueba chi-cuadrado global
table = df_filtered[["D. suzukii", "D. melanogaster"]].T.to_numpy()
chi2, p_global, dof, expected = chi2_contingency(table)

print(f"Chi² global = {chi2:.2f}, p-value = {p_global:.6f}, df = {dof}")

# 🔹 Pruebas post-hoc: test exacto de Fisher por superfamilia
pvals = []
labels = []

for _, row in df_filtered.iterrows():
    table_2x2 = [[row["D. suzukii"], row["D. melanogaster"]],
                 [df_filtered["D. suzukii"].sum() - row["D. suzukii"],
                  df_filtered["D. melanogaster"].sum() - row["D. melanogaster"]]]
    _, p = fisher_exact(table_2x2)
    pvals.append(p)
    labels.append(row["Superfamily"])

# 🔹 Corrección de Bonferroni
reject, pvals_corr, _, _ = multipletests(pvals, method='bonferroni')

# 🔹 Mostrar resultados
print("\nPost-hoc pairwise Fisher tests (Bonferroni corrected):")
for label, p_raw, p_adj, sig in zip(labels, pvals, pvals_corr, reject):
    print(f"{label}: p = {p_raw:.4f}, p_corr = {p_adj:.4f}, significant: {sig}")
